# Ingesta Masiva: 2015-2025 Parquet a Snowflake (RAW)

Este notebook recorre en un **solo bucle continuo** desde 2015 a 2025 para todos los meses,
descargando de a 1 archivo a la vez (procesamiento por lotes/batches por cada mes)
para evitar sobrecargar la memoria de la computadora local. Cada archivo es leído por PySpark,
se inyectan metadatos, se sube a Snowflake en modo `append` y luego se **elimina** localmente.

In [12]:
# 1. Configuración de librerías y de PySpark
%pip install python-dotenv
import os
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
from dotenv import load_dotenv

# Descubre el archivo .env que está en el directorio padre
load_dotenv('../.env')

# Inicializamos Spark empaquetando los conectores necesarios hacia Snowflake
spark = (
    SparkSession.builder
    .appName("Ingesta_Masiva_2015_2025")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "net.snowflake:spark-snowflake_2.12:2.12.0-spark_3.4,net.snowflake:snowflake-jdbc:3.14.4"
    )
    .getOrCreate()
)

# Silenciamos logs excesivos en consola para facilitar la lectura
spark.sparkContext.setLogLevel("WARN")

Note: you may need to restart the kernel to use updated packages.


In [13]:
print(f"Número de particiones: {df.rdd.getNumPartitions()}")

Número de particiones: 24


In [14]:
# 2. Opciones de conexión a Snowflake (Credenciales)
sfOptions = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": os.getenv("SNOWFLAKE_DATABASE"),
    "sfSchema": os.getenv("SNOWFLAKE_SCHEMA_RAW", "RAW"),
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

# Validamos el RUN_ID, por defecto le damos uno de run_local si está vacío.
run_id = os.getenv("RUN_ID", "run_local_mac_")

print("bloque ejecutado")

bloque ejecutado


In [15]:
# 3. Bucle Principal Multi-Anual (Batches por Mes)
años = range(2015, 2026)      # Loop desde 2015 hasta 2025 inclusive
meses = range(1, 13)         # Loop de meses 1 al 12
colores = ["yellow", "green"]  # Las dos tablas especificadas

# Guardamos en la carpeta data/ (o /tmp/) del contenedor un solo parquet temporal a la vez
temp_parquet = "../data/current_batch.parquet"

for color in colores:
    # Garantizamos nombres de tablas separados para yellow y green
    table_dest = "YELLOW_TRIPS_RAW" if color == "yellow" else "GREEN_TRIPS_RAW"
    
    for year in años:
        for month in meses:
            month_str = f"{month:02d}"
            url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month_str}.parquet"
            
            print(f"\n>>> Iniciando Ingesta: {color.upper()} {year}-{month_str}")
            
            try:
                # Paso 1: Descargar el archivo del mes actual. Esto dosifica la ingesta y no llena tu memoria
                urllib.request.urlretrieve(url, temp_parquet)
            except Exception as e:
                print(f"  [!] Archivo no encontrado o fallo al descargar (URL: {url}). Error: {e}")
                continue
            
            try:
                # Paso 2: PySpark lee dicho archivo en memoria (un Dataframe a la vez)
                df = spark.read.parquet(temp_parquet)
                
                # Convertir TimestampNTZ a Timestamp (Compatibilidad con Snowflake Connector)
                from pyspark.sql.types import TimestampNTZType, TimestampType
                for field in df.schema.fields:
                    if isinstance(field.dataType, TimestampNTZType):
                        df = df.withColumn(field.name, df[field.name].cast(TimestampType()))
                
                # Paso 3: Agregando la Metadata Literal como lo dicta el PDF (run_id, year, month, etc)
                df_with_meta = df.withColumn("run_id", lit(run_id)) \
                                 .withColumn("source_year", lit(year)) \
                                 .withColumn("source_month", lit(month)) \
                                 .withColumn("service_type", lit(color)) \
                                 .withColumn("ingested_at_utc", current_timestamp()) 
                
                # Paso 4: Escritura hacia nuestro Snowflake RAW utilizando API directa pyspark-snowflake.
                
                df_with_meta.write.format("snowflake") \
                            .options(**sfOptions) \
                            .option("dbtable", table_dest) \
                            .mode("append") \
                            .save()
                
                print(f"  [v] Batch insertado de forma exitosa en -> {table_dest}")
                
            except Exception as read_err:
                print(f"  [!] Fallo durante lectura de df o ingesta web en {color} {year}-{month_str}. Error: {read_err}")
                
            finally:
                # Paso 5: Cleanup! Borrar archivo parquet temporal recién usado para alivianar disco antes del siguiente mes.
                if os.path.exists(temp_parquet):
                    os.remove(temp_parquet)
                    
print("\n=======================================")
print("🎉 INGESTA MASIVA DE 10 AÑOS FINALIZADA 🎉")
print("=======================================")


>>> Iniciando Ingesta: YELLOW 2015-01
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-02
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-03
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-04
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-05
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-06
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-07
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-08
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-09
  [v] Batch insertado de forma exitosa en -> YELLOW_TRIPS_RAW

>>> Iniciando Ingesta: YELLOW 2015-10
  [v] Batch insertado de forma exitosa en -> YELLOW_

### Reintento de Meses Fallidos
- **YELLOW 2025**: Fallaron porque a partir de 2025 la estructura del parquet de taxis amarillos agregó una columna extra, rompiendo la compatibilidad de 24 columnas exactas.
- **GREEN finales de 2024 y 2025**: Fallaron con 403 Forbidden porque esa información sencillamente aún no ha sido compartida publicamente de forma abierta.

In [17]:
# Lista estricta de meses que fallaron (se puede modificar si te fallan, estoy meses fallron porque tiene una fila extra)
failed_batches = [
    #("yellow", 2025, range(1, 13)),
    #("green", 2024, range(10, 13)),
    ("green", 2025, range(1, 13))
]

for color, year, meses in failed_batches:
    table_dest = "YELLOW_TRIPS_RAW" if color == "yellow" else "GREEN_TRIPS_RAW"
    for month in meses:
        month_str = f"{month:02d}"
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month_str}.parquet"
        print(f"\n>>> Reintentando: {color.upper()} {year}-{month_str}")
        
        try:
            urllib.request.urlretrieve(url, temp_parquet)
        except Exception as e:
            print(f"  [X] Sigue sin existir (403 Forbidden / No publicado) -> {url}")
            continue
        
        try:
            df = spark.read.parquet(temp_parquet)
            
            # Solución para YELLOW 2025 y GREEN 2025: Evitar la nueva columna extra de 2025
            if color == "yellow" and year >= 2025:
                columnas_originales = df.columns[:19] 
                df = df.select(*columnas_originales)
            elif color == "green" and year >= 2025:
                columnas_originales = df.columns[:20] 
                df = df.select(*columnas_originales)
            
            from pyspark.sql.types import TimestampNTZType, TimestampType
            for field in df.schema.fields:
                if isinstance(field.dataType, TimestampNTZType):
                    df = df.withColumn(field.name, df[field.name].cast(TimestampType()))
            
            df_with_meta = df.withColumn("run_id", lit(run_id)) \
                             .withColumn("source_year", lit(year)) \
                             .withColumn("source_month", lit(month)) \
                             .withColumn("service_type", lit(color)) \
                             .withColumn("ingested_at_utc", current_timestamp())
            
            # Aseguramos idempotencia: eliminamos filas pre-existentes de ese mes antes de anexar
            delete_query = f"DELETE FROM {table_dest} WHERE source_year = {year} AND source_month = {month}"
            
            df_with_meta.write.format("snowflake") \
                        .options(**sfOptions) \
                        .option("dbtable", table_dest) \
                        .option("preactions", delete_query) \
                        .mode("append") \
                        .save()
            
            print(f"  [v] Logrado y subido a -> {table_dest}")
        except Exception as read_err:
            print(f"  [!] Fallo con error: {read_err}")
        finally:
            if os.path.exists(temp_parquet):
                os.remove(temp_parquet)



>>> Reintentando: GREEN 2025-01
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-02
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-03
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-04
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-05
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-06
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-07
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-08
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-09
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-10
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-11
  [v] Logrado y subido a -> GREEN_TRIPS_RAW

>>> Reintentando: GREEN 2025-12
  [v] Logrado y subido a -> GREEN_TRIPS_RAW


In [18]:
query_auditoria = """
SELECT 
    'YELLOW' AS service_type,
    source_year,
    source_month,
    COUNT(*) as total_trips
FROM YELLOW_TRIPS_RAW
GROUP BY source_year, source_month

UNION ALL

SELECT 
    'GREEN' AS service_type,
    source_year,
    source_month,
    COUNT(*) as total_trips
FROM GREEN_TRIPS_RAW
GROUP BY source_year, source_month

ORDER BY service_type DESC, source_year ASC, source_month ASC
"""

# Enviamos la query directamente a Snowflake a través de Spark
df_audit = spark.read.format("snowflake") \
              .options(**sfOptions) \
              .option("query", query_auditoria) \
              .load()

# Mostramos el resultado (usamos n=300 para no ocultar ningún mes, ya que son como 250 filas)
df_audit.show(n=300, truncate=False)


+------------+-----------+------------+-----------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|TOTAL_TRIPS|
+------------+-----------+------------+-----------+
|YELLOW      |2017       |10          |9768672    |
|YELLOW      |2017       |11          |9284803    |
|YELLOW      |2017       |12          |9508501    |
|YELLOW      |2018       |1           |8760687    |
|YELLOW      |2018       |2           |8492819    |
|YELLOW      |2018       |3           |9431289    |
|YELLOW      |2018       |4           |9306216    |
|YELLOW      |2018       |5           |9224788    |
|YELLOW      |2018       |6           |8714667    |
|YELLOW      |2018       |7           |7851143    |
|YELLOW      |2018       |8           |7855040    |
|YELLOW      |2018       |9           |8049094    |
|YELLOW      |2018       |10          |8834520    |
|YELLOW      |2018       |11          |8155449    |
|YELLOW      |2018       |12          |8195675    |
|YELLOW      |2019       |1           |7696617    |
|YELLOW     